In [34]:
%pip install pandas openpyxl pyarrow matplotlib seaborn scikit-learn statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import pandas as pd

df = pd.read_excel("../data/Modified_loan_dataset_investor.xlsx")
df.to_parquet("../data/loans.parquet")   # use this from now on

In [36]:
df.shape


(843755, 22)

In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 843755 entries, 0 to 843754
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       843755 non-null  str           
 1   country                       843755 non-null  str           
 2   loan_issued_at                843755 non-null  datetime64[us]
 3   early_repaid_at               252308 non-null  object        
 4   issued_amount                 843755 non-null  float64       
 5   loan_status                   843755 non-null  str           
 6   initial_interest_rate         843755 non-null  float64       
 7   nr_of_payments                843755 non-null  int64         
 8   principal_balance             836042 non-null  float64       
 9   principal_debt                836042 non-null  float64       
 10  principal_paid_total          836042 non-null  float64       
 11  interest_paid_total     

In [38]:
print(df["is_default"].unique())

[ 0.  1. nan]


In [39]:
print(df["has_default_within_12_months"].unique())

[ 0. nan  1.]


In [40]:
print(df["customer_risk_rating"].unique())

<ArrowStringArray>
[nan, 'D', 'C', 'B', 'AA', 'F', 'A', 'E', 'G', 'HR']
Length: 10, dtype: str


In [41]:
df["is_default"] = df["is_default"].astype("Int8")
df["has_default_within_12_months"] = df["has_default_within_12_months"].astype("Int8")

for col in ["country", "loan_status", "loan_status_risk", "customer_risk_rating"]:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 843755 entries, 0 to 843754
Data columns (total 22 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   loan_id                       843755 non-null  str           
 1   country                       843755 non-null  category      
 2   loan_issued_at                843755 non-null  datetime64[us]
 3   early_repaid_at               252308 non-null  object        
 4   issued_amount                 843755 non-null  float64       
 5   loan_status                   843755 non-null  category      
 6   initial_interest_rate         843755 non-null  float64       
 7   nr_of_payments                843755 non-null  int64         
 8   principal_balance             836042 non-null  float64       
 9   principal_debt                836042 non-null  float64       
 10  principal_paid_total          836042 non-null  float64       
 11  interest_paid_total     

In [42]:
order = ["AA", "A", "B", "C", "D", "E", "F", "G", "HR"]  # replace with your actual values
df["customer_risk_rating"] = pd.Categorical(df["customer_risk_rating"], categories=order, ordered=True)

**customer_risk_rating grades each customer at the time the loan was issued. **AA is the best grade** (the most reliable customers, least likely to default) and **HR is the lowest grade** (the riskiest customers, most likely to default)**.

**We converted the column to an **ordered category**, from best to lowest grade:**

**AA → A → B → C → D → E → F → G → HR**

In [43]:
df = df.drop(columns=["nr_of_payments"])

In [44]:
df.shape

(843755, 21)

**nr_of_payments was removed because it is only filled on the day a payment is due, so its value depends on the data extraction date rather than on the loan itself. It carries no information about default risk, and months_on_book already captures how far along each loan is**


In [45]:
df["is_default"].isna().sum()

np.int64(7713)

In [46]:
df["combined_income"].isna().sum()

np.int64(273191)

In [47]:
df["customer_risk_rating"].isna().sum()


np.int64(338380)

In [48]:
df["has_default_within_12_months"].isna().sum()

np.int64(188117)

In [49]:
df["months_in_default"].isna().sum()

np.int64(727045)

**The total dataset is (789144), but when we check here at months_in_defalt, it shows that missing values are (681608). But none is not default. Blank means “not in default” rather than “unknown”. As per the dataset dictionary, the number of months since default occurred, NULL if not default. Amount of months in default.**

In [51]:
df.describe()

,loan_issued_at,issued_amount,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,interest_paid_total,is_default,months_in_default,months_on_book,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months
count,843755,843755.000000,843755.000000,836042.000000,836042.000000,836042.000000,836042.000000,836042.0,116710.000000,834049.000000,836042.000000,843755.000000,570564.000000,655638.0
mean,2023-06-02 10:07:04.604132,2225.808536,0.213592,1001.797883,268.140680,1089.300695,480.002451,0.139598,33.228824,16.136241,1651.533920,58.014713,2556.550101,0.13827
min,2009-02-28 18:05:13,6.390000,0.000000,-34.200000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.0
25%,2021-11-26 11:30:02,519.000000,0.174800,0.000000,0.000000,95.860000,32.500000,0.0,8.000000,5.000000,213.110000,36.000000,1697.970000,0.0
50%,2024-07-05 12:07:45.087000,1275.000000,0.185000,101.285000,0.000000,436.880000,149.840000,0.0,20.000000,11.000000,711.180000,60.000000,2356.665000,0.0
75%,2025-09-08 12:19:23.746500,3508.000000,0.217000,1068.270000,0.000000,1349.000000,542.770000,0.0,49.000000,22.000000,2225.440000,60.000000,3058.957807,0.0
max,2026-09-30 02:50:06.623000,30000.000000,2.643100,29965.050000,19744.280000,30000.000000,34494.530000,1.0,202.000000,161.000000,41304.710000,180.000000,177871.700000,1.0
std,NaN,2442.874706,0.131016,1981.502380,979.444736,1670.075454,904.139309,0.34657,32.838894,16.347041,2304.241191,31.660512,1542.805530,0.345183


In [52]:
df.isna().mean().sort_values(ascending=False)   # missing values in Percentage in Descending order

months_in_default               0.861678
early_repaid_at                 0.700970
next_payment_date_local         0.589351
customer_risk_rating            0.401041
combined_income                 0.323780
has_default_within_12_months    0.222952
months_on_book                  0.011503
is_default                      0.009141
interest_paid_total             0.009141
principal_balance               0.009141
repaid_amount_total             0.009141
principal_paid_total            0.009141
principal_debt                  0.009141
loan_status_risk                0.000002
loan_issued_at                  0.000000
country                         0.000000
loan_id                         0.000000
issued_amount                   0.000000
loan_status                     0.000000
initial_interest_rate           0.000000
initial_loan_duration           0.000000
dtype: float64

In [53]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0].sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
months_in_default,727045,86.2
early_repaid_at,591447,70.1
next_payment_date_local,497268,58.9
customer_risk_rating,338380,40.1
combined_income,273191,32.4
has_default_within_12_months,188117,22.3
months_on_book,9706,1.2
is_default,7713,0.9
principal_paid_total,7713,0.9
principal_debt,7713,0.9


In [54]:
df[df["is_default"].isna()]    # Comparing is_Default column related to others columns to see if there is any pattern in missing values

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,is_default,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating
333,01EB1FD2-0510-4B7F-9E5E-A4B800BC9FF5,Finland,2015-06-15 23:05:15,2015-06-16 09:17:38,3000.0,Returned,0.1872,NaN,NaN,NaN,...,<NA>,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
483,0509BABF-17D9-4EB4-9E67-A4B2017192AA,Spain,2015-06-13 11:40:49,2015-06-15 10:48:34,1000.0,Returned,0.5709,NaN,NaN,NaN,...,<NA>,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN
574,06E7E6D8-48FB-487C-A054-A38901492C80,Spain,2014-08-20 09:00:00,2014-08-25 13:20:01,1000.0,Repaid,0.3400,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Cancelled,NaN,12,NaN,0,NaN
611,07BEF4BA-CF83-49ED-AE33-A3CA00E01BD9,Spain,2014-10-24 16:55:09,2014-11-21 01:30:14,1800.0,Repaid,0.3100,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN
653,08A3A0DA-46FE-45F0-BA3C-3EB2F1CCFF50,Estonia,2015-04-12 17:30:00,2015-04-29 01:30:05,500.0,Repaid,0.2596,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,48,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843373,F80FB4C9-74D5-407D-961E-2B24D4DCED9C,Estonia,2015-03-17 02:40:07,42103.062581,500.0,Repaid,0.4290,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,12,NaN,0,NaN
843479,FA5C28B4-6C89-4587-8EB0-A4EC011F37FE,Spain,2015-08-12 10:15:32,42236.062685,2000.0,Repaid,0.3112,NaN,NaN,NaN,...,<NA>,NaN,NaN,NaN,Cancelled,NaN,48,NaN,0,NaN
843514,FAF4AA2C-C74B-4EB9-9B79-A37F013A61D9,Slovakia,2014-08-08 20:20:00,41879.572118,1400.0,Repaid,0.2900,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN
843648,FD6D4B81-B3F6-447F-ABAD-A4040152641E,Spain,2014-12-27 21:00:00,42028.062593,800.0,Repaid,0.3500,NaN,NaN,NaN,...,<NA>,NaN,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN


In [55]:
df["default_flag"] = df["is_default"].fillna(0).astype(int)    

**We changed the missing values in is_default to 0, so that we can use it for analysis. We will use this column for analysis instead of is_default column**.

**Because in **Loan_Status** all of the Borrowers who have missing values in the **is_default** column repaid or returned the money back. so the missing values in the is_default column doesn't mean missing. So we changed missing values to 0 which is not Default and they paid back the money**.

**We created a new default_flag column using only '0' and '1'** **( 0 = not default, 1 = default)**

In [56]:
print(df["is_default"].value_counts(dropna=False))
print(df["default_flag"].value_counts())            # counted the values of the new column to see if it is correct and matches the original column

is_default
0       719332
1       116710
<NA>      7713
Name: count, dtype: Int64
default_flag
0    727045
1    116710
Name: count, dtype: int64


**Combined_income - Borrower's income, combined from bank statement and registry data (tax)**

**The issues amount for the loan_id 789131 is 2125 euros, for this person the combined income is NaN( Missing ), so we need to delete the missing values row even though the issued amount is high.**

In [57]:
df[df["combined_income"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
0,0001981E-E281-499C-906E-AB6100D6BF73,Estonia,2020-02-14 13:10:19,2021-03-01 08:41:52,637.0,Repaid,0.2866,0.00,0.00,637.00,...,NaN,NaN,12.0,Paid Up,831.70,36,NaN,0,NaN,0
2,000E567F-C487-4E02-805D-A2D101020413,Estonia,2014-02-15 09:40:02,NaN,7800.0,Repaid,0.3000,0.00,0.00,7800.00,...,NaN,NaN,62.0,Paid Up,15291.57,36,NaN,0,NaN,0
6,001706E1-FC54-4D75-8630-AD30007D53EA,Estonia,2021-05-22 09:06:56,NaN,5209.0,Defaulted,0.3438,2188.50,2188.50,3020.50,...,NaN,51.0,13.0,Defaulted,4421.31,60,NaN,0,NaN,1
8,001FE05D-A847-45A5-B842-AA750150A417,Estonia,2019-06-23 20:30:26,NaN,1060.0,Repaid,0.3107,0.00,0.00,355.26,...,NaN,NaN,18.0,Written Off,789.83,60,NaN,0,NaN,0
16,0037ED6E-835F-4962-BBBC-AB4B00698CEE,Finland,2020-01-23 20:21:39,NaN,4150.0,Defaulted,0.2000,3422.04,3422.04,727.96,...,NaN,71.0,10.0,Defaulted,1057.67,60,NaN,1,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843742,FF9EED08-776C-4DE0-92E0-AE7F01446FBA,Finland,2022-04-22 19:46:41,NaN,1556.0,Repaid,0.1927,0.00,0.00,1177.38,...,NaN,NaN,15.0,Written Off,1415.60,60,NaN,0,NaN,0
843745,FFD3535E-61E0-41C8-A124-ADFB017F6C37,Finland,2021-12-12 01:29:58,NaN,518.0,Repaid,0.1927,0.00,0.00,518.00,...,NaN,NaN,2.0,Paid Up,528.07,3,NaN,0,NaN,0
843749,FFE302F6-3D53-495B-8C21-A86800AF311E,Estonia,2018-01-14 10:49:42,NaN,1485.0,Repaid,0.1957,0.00,0.00,1054.90,...,NaN,NaN,4.0,Paid Up,1054.90,60,NaN,1,NaN,0
843751,FFEFC505-692D-4DD2-9C06-AF2D00B5097C,Finland,2022-10-13 11:24:14,45079.916933,4150.0,Repaid,0.1927,0.00,0.00,4150.00,...,NaN,NaN,7.0,Paid Up,4639.52,60,NaN,0,NaN,0


In [59]:
df[df["customer_risk_rating"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
0,0001981E-E281-499C-906E-AB6100D6BF73,Estonia,2020-02-14 13:10:19,2021-03-01 08:41:52,637.0,Repaid,0.2866,0.0,0.0,637.00,...,NaN,NaN,12.0,Paid Up,831.70,36,NaN,0,NaN,0
1,0009E1E6-1880-44DE-9F3F-AFE9011E58C7,Finland,2023-04-19 18:17:39,NaN,4146.0,Repaid,0.1927,0.0,0.0,2438.49,...,NaN,NaN,15.0,Written Off,3010.50,60,2124.89,0,NaN,0
2,000E567F-C487-4E02-805D-A2D101020413,Estonia,2014-02-15 09:40:02,NaN,7800.0,Repaid,0.3000,0.0,0.0,7800.00,...,NaN,NaN,62.0,Paid Up,15291.57,36,NaN,0,NaN,0
5,0014F40C-0920-4B6A-898B-AFC1001E4CE0,Finland,2023-03-10 01:54:53,2025-02-19 11:48:13.287000,4146.0,Repaid,0.1927,0.0,0.0,4146.00,...,NaN,NaN,23.0,Paid Up,5676.59,24,3211.36,0,NaN,0
6,001706E1-FC54-4D75-8630-AD30007D53EA,Estonia,2021-05-22 09:06:56,NaN,5209.0,Defaulted,0.3438,2188.5,2188.5,3020.50,...,NaN,51.0,13.0,Defaulted,4421.31,60,NaN,0,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843742,FF9EED08-776C-4DE0-92E0-AE7F01446FBA,Finland,2022-04-22 19:46:41,NaN,1556.0,Repaid,0.1927,0.0,0.0,1177.38,...,NaN,NaN,15.0,Written Off,1415.60,60,NaN,0,NaN,0
843745,FFD3535E-61E0-41C8-A124-ADFB017F6C37,Finland,2021-12-12 01:29:58,NaN,518.0,Repaid,0.1927,0.0,0.0,518.00,...,NaN,NaN,2.0,Paid Up,528.07,3,NaN,0,NaN,0
843749,FFE302F6-3D53-495B-8C21-A86800AF311E,Estonia,2018-01-14 10:49:42,NaN,1485.0,Repaid,0.1957,0.0,0.0,1054.90,...,NaN,NaN,4.0,Paid Up,1054.90,60,NaN,1,NaN,0
843751,FFEFC505-692D-4DD2-9C06-AF2D00B5097C,Finland,2022-10-13 11:24:14,45079.916933,4150.0,Repaid,0.1927,0.0,0.0,4150.00,...,NaN,NaN,7.0,Paid Up,4639.52,60,NaN,0,NaN,0


**Customer_Risk_Rating - The rating system may start after a specific date not at the same day they lend the money. According to date of **loan_issued_at**, Many borrowers who borrowed money earlier and repaid the money back but still in the customer-risk_rating column it shows NaN**

In [60]:
df[df["has_default_within_12_months"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
7,0019B23E-C261-4442-A6CB-B48D01762F58,Netherlands,2026-07-20 22:47:55.220,NaN,728.0,Active,0.0715,671.22,0.0,56.78,...,2026-11-02 00:00:00,NaN,3.0,Active,67.03,24,3683.54,<NA>,C,0
9,00205A14-158D-4FA3-B911-B4CE011C8229,Finland,2026-09-23 17:24:54.910,NaN,500.0,Active,0.1748,500.00,0.0,0.00,...,2026-11-04 00:00:00,NaN,0.0,Active,0.00,12,2688.89,<NA>,B,0
15,0034D23F-FCBD-4018-AECA-B3DB00CA16FF,Estonia,2026-01-23 12:46:20.803,NaN,318.0,Active,0.2219,161.59,0.0,156.41,...,2026-11-09 00:00:00,NaN,9.0,Active,164.05,18,2000.00,<NA>,B,0
21,00467D8C-56DE-4B57-AF0D-B422017ED706,Finland,2026-04-05 07:00:47.730,NaN,500.0,Active,0.1748,458.72,0.0,41.28,...,2026-11-17 00:00:00,NaN,6.0,Active,95.48,24,2934.41,<NA>,AA,0
22,004931B4-AFE3-461E-9B61-B464012FD2DB,Estonia,2026-06-09 18:42:16.243,NaN,1063.0,Active,0.2539,1044.16,0.0,18.84,...,2026-11-10 00:00:00,NaN,4.0,Active,109.70,84,658.29,<NA>,AA,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843733,FF79C563-563F-4FD1-81F9-B42700D648B7,Finland,2026-04-09 14:22:06.827,NaN,100.0,Active,0.1748,98.05,0.0,1.95,...,46342,NaN,6.0,Active,12.77,120,2055.72,<NA>,A,0
843737,FF915E11-EB75-4F8E-9FB4-B44A016684B8,Finland,2026-05-14 21:51:07.927,NaN,300.0,Active,0.1748,294.03,0.0,5.97,...,46331,NaN,5.0,Active,30.62,108,1491.13,<NA>,AA,0
843738,FF94A3DD-83D8-474D-BE86-B49B00EDBDFB,Finland,2026-08-03 14:30:31.630,NaN,500.0,Active,0.1748,493.81,0.0,6.19,...,46343,NaN,2.0,Active,27.86,84,3087.58,<NA>,D,0
843744,FFBE7CB4-02B8-4977-8BAE-B40E00B0E81A,Finland,2026-03-15 10:54:53.733,46246.528307,1000.0,Repaid,0.1748,0.00,0.0,1000.00,...,NaN,NaN,4.0,Paid Up,1085.58,48,2602.40,<NA>,AA,0


**The Borrowers lend money from investors in 2026 but has_default_within_12 Months showing NaN (Missing Values). The dataset is published in 26th oct 2026. So for those borrowers having NaN still means No values for this Columns. Should we delete it. ( Deleting the rows which ever dates having the year 2026)**

In [61]:
df[df["repaid_amount_total"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
333,01EB1FD2-0510-4B7F-9E5E-A4B800BC9FF5,Finland,2015-06-15 23:05:15,2015-06-16 09:17:38,3000.0,Returned,0.1872,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
483,0509BABF-17D9-4EB4-9E67-A4B2017192AA,Spain,2015-06-13 11:40:49,2015-06-15 10:48:34,1000.0,Returned,0.5709,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
574,06E7E6D8-48FB-487C-A054-A38901492C80,Spain,2014-08-20 09:00:00,2014-08-25 13:20:01,1000.0,Repaid,0.3400,NaN,NaN,NaN,...,NaN,NaN,0.0,Cancelled,NaN,12,NaN,0,NaN,0
611,07BEF4BA-CF83-49ED-AE33-A3CA00E01BD9,Spain,2014-10-24 16:55:09,2014-11-21 01:30:14,1800.0,Repaid,0.3100,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
653,08A3A0DA-46FE-45F0-BA3C-3EB2F1CCFF50,Estonia,2015-04-12 17:30:00,2015-04-29 01:30:05,500.0,Repaid,0.2596,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,48,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843373,F80FB4C9-74D5-407D-961E-2B24D4DCED9C,Estonia,2015-03-17 02:40:07,42103.062581,500.0,Repaid,0.4290,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,12,NaN,0,NaN,0
843479,FA5C28B4-6C89-4587-8EB0-A4EC011F37FE,Spain,2015-08-12 10:15:32,42236.062685,2000.0,Repaid,0.3112,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,48,NaN,0,NaN,0
843514,FAF4AA2C-C74B-4EB9-9B79-A37F013A61D9,Slovakia,2014-08-08 20:20:00,41879.572118,1400.0,Repaid,0.2900,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
843648,FD6D4B81-B3F6-447F-ABAD-A4040152641E,Spain,2014-12-27 21:00:00,42028.062593,800.0,Repaid,0.3500,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN,0


**They repaid even before one year, but in repaid_amount_total, we have as missing values. So can we delete this?**

In [62]:
df[df["principal_balance"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
333,01EB1FD2-0510-4B7F-9E5E-A4B800BC9FF5,Finland,2015-06-15 23:05:15,2015-06-16 09:17:38,3000.0,Returned,0.1872,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
483,0509BABF-17D9-4EB4-9E67-A4B2017192AA,Spain,2015-06-13 11:40:49,2015-06-15 10:48:34,1000.0,Returned,0.5709,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
574,06E7E6D8-48FB-487C-A054-A38901492C80,Spain,2014-08-20 09:00:00,2014-08-25 13:20:01,1000.0,Repaid,0.3400,NaN,NaN,NaN,...,NaN,NaN,0.0,Cancelled,NaN,12,NaN,0,NaN,0
611,07BEF4BA-CF83-49ED-AE33-A3CA00E01BD9,Spain,2014-10-24 16:55:09,2014-11-21 01:30:14,1800.0,Repaid,0.3100,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
653,08A3A0DA-46FE-45F0-BA3C-3EB2F1CCFF50,Estonia,2015-04-12 17:30:00,2015-04-29 01:30:05,500.0,Repaid,0.2596,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,48,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843373,F80FB4C9-74D5-407D-961E-2B24D4DCED9C,Estonia,2015-03-17 02:40:07,42103.062581,500.0,Repaid,0.4290,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,12,NaN,0,NaN,0
843479,FA5C28B4-6C89-4587-8EB0-A4EC011F37FE,Spain,2015-08-12 10:15:32,42236.062685,2000.0,Repaid,0.3112,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,48,NaN,0,NaN,0
843514,FAF4AA2C-C74B-4EB9-9B79-A37F013A61D9,Slovakia,2014-08-08 20:20:00,41879.572118,1400.0,Repaid,0.2900,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
843648,FD6D4B81-B3F6-447F-ABAD-A4040152641E,Spain,2014-12-27 21:00:00,42028.062593,800.0,Repaid,0.3500,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN,0


**The Borrowers, they pay back within one month, so there is no money to pay back and principal_balance is the balance of money to pay back so this null values is meaningless, we should remove.**

In [63]:
df[df["principal_debt"].isna()]

,loan_id,country,loan_issued_at,early_repaid_at,issued_amount,loan_status,initial_interest_rate,principal_balance,principal_debt,principal_paid_total,...,next_payment_date_local,months_in_default,months_on_book,loan_status_risk,repaid_amount_total,initial_loan_duration,combined_income,has_default_within_12_months,customer_risk_rating,default_flag
333,01EB1FD2-0510-4B7F-9E5E-A4B800BC9FF5,Finland,2015-06-15 23:05:15,2015-06-16 09:17:38,3000.0,Returned,0.1872,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
483,0509BABF-17D9-4EB4-9E67-A4B2017192AA,Spain,2015-06-13 11:40:49,2015-06-15 10:48:34,1000.0,Returned,0.5709,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,60,NaN,0,NaN,0
574,06E7E6D8-48FB-487C-A054-A38901492C80,Spain,2014-08-20 09:00:00,2014-08-25 13:20:01,1000.0,Repaid,0.3400,NaN,NaN,NaN,...,NaN,NaN,0.0,Cancelled,NaN,12,NaN,0,NaN,0
611,07BEF4BA-CF83-49ED-AE33-A3CA00E01BD9,Spain,2014-10-24 16:55:09,2014-11-21 01:30:14,1800.0,Repaid,0.3100,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
653,08A3A0DA-46FE-45F0-BA3C-3EB2F1CCFF50,Estonia,2015-04-12 17:30:00,2015-04-29 01:30:05,500.0,Repaid,0.2596,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,48,NaN,0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843373,F80FB4C9-74D5-407D-961E-2B24D4DCED9C,Estonia,2015-03-17 02:40:07,42103.062581,500.0,Repaid,0.4290,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,12,NaN,0,NaN,0
843479,FA5C28B4-6C89-4587-8EB0-A4EC011F37FE,Spain,2015-08-12 10:15:32,42236.062685,2000.0,Repaid,0.3112,NaN,NaN,NaN,...,NaN,NaN,NaN,Cancelled,NaN,48,NaN,0,NaN,0
843514,FAF4AA2C-C74B-4EB9-9B79-A37F013A61D9,Slovakia,2014-08-08 20:20:00,41879.572118,1400.0,Repaid,0.2900,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,60,NaN,0,NaN,0
843648,FD6D4B81-B3F6-447F-ABAD-A4040152641E,Spain,2014-12-27 21:00:00,42028.062593,800.0,Repaid,0.3500,NaN,NaN,NaN,...,NaN,NaN,0.0,Paid Up,NaN,36,NaN,0,NaN,0


**For all of the Borrowers, they returned the money. So we do not have principal_debt as numbers in this column, which make sense not to have it since they pay back. But for principal_balance and principal_paid_total, we should have some number that they repay. Here we do not have it, so this is really missing values. Should we delete this or how can we handle this?**